# NMR from First Principles with uniqx

NMR (Nuclear Magnetic Resonance) spectroscopy probes the local magnetic environment of
each nucleus in a molecule. Computing an NMR spectrum from first principles requires
three coupled steps:

1. **SCF (Hartree-Fock)** — solve for the molecular orbital coefficients and electronic
   density.
2. **Magnetic shieldings (GIAO + CPHF)** — compute the chemical-shift tensor at every
   nucleus by perturbing the SCF problem with an external magnetic field.
3. **J-couplings (Fermi contact)** — compute the indirect spin-spin coupling between
   pairs of nuclei via the contact interaction at the nuclear position.

This notebook traces the entire pipeline as a single computation graph and submits it
to uniqx for execution. No PySCF or other external chemistry library is required —
all SCF / CPHF / response computations run server-side.

## 1. Setup

Imports and connection. Set `UNIQX_GATEWAY` to point at your service; defaults to
`api.oriqx.com:443` (the hosted hackathon gateway); use `localhost:50050` only for local dev.

In [ ]:
import os
import math

from uniqx import connect, submit, get, preflight, parse_buffer_view, login
from uniqx.domains.chemistry.basis import extract_basis
from uniqx.domains.chemistry.nmr_full import nmr_full_module

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)

## 2. Define a Molecule

We use water at the equilibrium geometry (O-H bond length 0.96 Å, H-O-H angle 104.5°).
Geometries are passed as a list of `(element, [x, y, z])` tuples in Angstrom.

In [ ]:
H2O = [
    ("O", [0.0,  0.0,     0.1173]),
    ("H", [0.0,  0.7572, -0.4692]),
    ("H", [0.0, -0.7572, -0.4692]),
]

info = extract_basis(H2O, "sto-3g")
print(f"basis:        sto-3g")
print(f"n_atoms:      {info.n_atoms}")
print(f"n_basis:      {info.n_basis}")
print(f"n_electrons:  {info.n_electrons}")
print(f"E_nuc:        {info.nuclear_repulsion:.6f} Ha")

## 3. Trace the Full NMR Pipeline

`nmr_full_module` returns a traced module with three outputs:

- **SCF energy** — the converged total electronic + nuclear repulsion energy (Hartree).
- **Isotropic shieldings** — one number per atom (ppm). For ¹H the chemical shift
  relative to TMS is `δ = σ_TMS - σ_atom`.
- **Fermi-contact J-couplings** — one entry per atom pair, in Hz.

The Python call below does *not* execute the SCF or CPHF steps — it traces them into
a computation graph that uniqx will compile and execute.

In [ ]:
mod = nmr_full_module(H2O, info, max_scf_iter=50, max_cphf_iter=20)

fn = mod.functions[0]
print(f"module:    {mod.name}")
print(f"functions: {len(mod.functions)}")
print(f"ops:       {len(fn.ops)}")
print(f"params:    {len(fn.params)} (basis flat lists)")
print(f"results:   {len(fn.results)} (scf_energy, shieldings, jcouplings)")

## 4. Preflight — Inspect Execution Options

Preflight asks uniqx to plan the computation across available hardware (CPU, GPU,
QPU). It returns a list of Pareto-optimal options trading off time, cost, error,
and carbon. We don't need to pick one explicitly here — the platform will use the
recommended option on submit.

In [ ]:
options = preflight(mod, client=client)
print(options.summary())

## 5. Submit and Wait for Results

Runtime inputs are flat numerical lists describing the molecule's basis-set and
geometry, in the order the traced module expects.

In [ ]:
runtime_inputs = [
    list(info.exps_flat),
    list(info.coeffs_flat),
    list(info.centers_flat),
    list(info.ang_flat),
    list(info.atom_coords_flat),
    list(info.charges_flat),
]

job_id = submit(mod, client=client, runtime_inputs=runtime_inputs)
print("job_id:", job_id)

result = get(job_id, client=client, timeout=300.0)
print("state:", result.get("state"))

## 6. Parse the Results

The result payload is a sequence of buffer-views — one per output of the traced
module. We decode them in order: SCF energy, shieldings, J-couplings.

In [ ]:
payload = result.get("payload") or result.get("result_payload") or b""
if isinstance(payload, str):
    payload = payload.encode("utf-8", errors="replace")

views = [parse_buffer_view(ln) for ln in payload.decode("utf-8", errors="replace").splitlines() if ln.strip()]
views = [v for v in views if v is not None]

expected = 3  # scf_energy, shieldings, jcouplings
if len(views) < expected:
    print(f"warning: expected {expected} output views, got {len(views)} — result is partial")
print(f"received {len(views)} output view(s)")
print()

if len(views) >= 1 and len(views[0][2]) >= 1:
    scf_energy = views[0][2][0]
    print(f"SCF energy: {scf_energy:.6f} Ha")
else:
    print("SCF energy: not available in this run")
print()

if len(views) >= 2:
    shieldings = views[1][2]
    print("Isotropic shieldings (ppm):")
    for atom, sigma in zip(H2O, shieldings):
        print(f"  {atom[0]:<2}  {sigma:>10.3f}")
else:
    print("Isotropic shieldings: not available in this run")
print()

if len(views) >= 3:
    jcouplings = views[2][2]
    print(f"J-coupling matrix entries: {len(jcouplings)}")
    print(f"  (atom-pair list, FC contribution only, in Hz)")
else:
    print("J-couplings: not available in this run")

## 7. Chemical Shifts vs. TMS

To get a chemical shift `δ` in ppm, subtract the absolute shielding `σ` from a
reference shielding (typically TMS = tetramethylsilane). Here we just report
`σ` directly — convert to `δ` by re-running the same module against TMS in your
chosen basis and method, then subtracting per-atom.

## Summary

You traced an SCF + CPHF + Fermi-contact pipeline as a single computation graph,
let uniqx pick the best execution option, and decoded the result.

What to try next:

- Swap `H2O` for `formaldehyde`, `methanol`, or your own geometry.
- Switch to a larger basis like `cc-pvdz` (extract_basis supports it).
- Use `shieldings_module` if you only need shieldings (faster — skips J-couplings).
- Run `preflight()` against a richer hardware mix to see GPU/QPU options.